# The trend ledger

The deflated Sharpe of every trend result, counting every backtest the lane ran. The late check ran about 5,000 backtests but was deflated as if it had not, so its results are rechecked here.

## Setup

Near-duplicate backtests count as less than one, so the total is given as a number of independent backtests.

The late check ran each strategy on all seven weekdays. Each version is its own backtest, but for counting, each week's return is dated to the Sunday that week ends on, so the seven versions line up and count as nearly one. The 30 backtests on the earlier ten-coin data have no returns here, so each counts as fully independent.

In [1]:
import warnings
import numpy as np
import pandas as pd
from qp_research import condition, costs, data, gate, panel
from qp_research.book import Book, Sliced, inverse_vol, gated as gated_weights
from qp_research.cv import halves
from qp_research.data import universe
from qp_research.ledger import Ledger
import signals
from signals import sharpe

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

led = Ledger('ledger', every='W-SUN')
OLD_TRIALS = 30
print(f'ledger holds {len(led.table())} trials before this run')

ledger holds 5226 trials before this run


## The gate scoring's backtests

Rerun the 125 backtests the gate scoring counted. The 32 lookback backtests are the main test, the rest are exploratory. The counts should match the scoring's 42 and 74.

In [2]:
p = panel.load()
W, LIQ, VOL, F = p.returns, p.liquidity, p.vol, p.funding
ladder = costs.fit(costs.sample(['2023-03-10', '2024-03-10', '2025-03-10', '2026-06-10'], p.liquidity))
tb = Book(p, costs.Taker(costs.FEES['usdm'], ladder, 1e5), ledger=led)
LOOKBACKS = [1, 2, 3, 4, 5, 6, 8, 12]

def btc(L):
    w = pd.DataFrame(0.0, index=W.index, columns=W.columns)
    w['BTCUSDT'] = np.sign(W['BTCUSDT'].rolling(L, min_periods=L).sum()).shift(1)
    return w

gate_names = []
def run(w, name, kind='exploratory'):
    gate_names.append(name)
    return tb.run(w, name=name, kind=kind).net

for L in LOOKBACKS:
    run(signals.per_coin(p, L, 1e6), f'per coin $1M L{L}', 'primary')
    run(signals.per_coin(p, L), f'per coin $10M L{L}', 'primary')
    run(signals.market(p, L), f'market $10M L{L}', 'primary')
    run(btc(L), f'BTC L{L}', 'primary')

per_coin_w, market_w = signals.per_coin(p, 4), signals.market(p, 4)
idx = tb.run(per_coin_w, record=False).index
cells = condition.cells(condition.market_conditions(p), idx)

def moderate(n_ret=4, n_vol=12, q=1 / 3):
    return (condition.past_cut(condition.strength(p, n_ret, n_vol), q) == 1).reindex(idx).fillna(False)

def signal_book(sig, name, floor=1e7, hi=np.inf):
    return run(inverse_vol(p, sig.shift(1), floor, hi), name)

def gated(w, on_mask, name):
    return run(gated_weights(w, on_mask), name)

mret = condition.market_return(p)
level = W.cumsum()
for L in [1, 2, 3, 4, 6, 8, 12]:
    resid = W.rolling(L, min_periods=L).sum().sub(mret.rolling(L, min_periods=L).sum(), axis=0)
    signal_book(np.sign(resid), f'residual L{L}')
    if L == 4:
        signal_book(-np.sign(resid), 'residual reversal L4')
for L in [1, 2, 4, 8, 12]:
    ret = W.rolling(L, min_periods=L).sum()
    signal_book(np.sign(ret - F.rolling(L, min_periods=L).sum()), f'funding-aware L{L}')
    brk = pd.DataFrame(0.0, index=W.index, columns=W.columns) \
        .mask(level >= level.rolling(L, min_periods=L).max(), 1.0).mask(level <= level.rolling(L, min_periods=L).min(), -1.0)
    signal_book(brk, f'breakout L{L}')
    signal_book(np.sign(ret), f'small coins L{L}', 1e6, 1e7)
    if L > 1:
        er = (ret.abs() / W.abs().rolling(L, min_periods=L).sum()).where((LIQ >= 1e7) & VOL.notna() & ret.notna())
        lo, hi = er.quantile(1 / 3, axis=1), er.quantile(2 / 3, axis=1)
        for name, m in [('low', er.le(lo, axis=0)), ('mid', er.gt(lo, axis=0) & er.lt(hi, axis=0)), ('high', er.ge(hi, axis=0))]:
            signal_book(np.sign(ret).where(m), f'cleanliness {name} L{L}')
for n, m in cells.items():
    gated(per_coin_w, m, f'per coin, {n}')
    gated(market_w, m, f'market, {n}')

majors = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'XRPUSDT', 'DOGEUSDT']
oi = data.open_interest(majors, '2020-09-01', '2026-09-01')
contracts = np.log(oi.resample('W-SUN').last().reindex(W.index)).sub(W[majors].cumsum(), axis=0)
for k, v in {'OI growth': contracts.diff(4).mean(axis=1).shift(1),
             'OI level': (contracts - contracts.rolling(26, min_periods=13).mean()).mean(axis=1).shift(1)}.items():
    lab = condition.thirds(v).reindex(idx)
    for t, name in enumerate(['low', 'mid', 'high']):
        gated(per_coin_w, lab == t, f'per coin, {k} {name}')

on = moderate()
for q in [0.25, 0.5]:
    gated(per_coin_w, moderate(q=q), f'moderate, cut {q:.0%}')
for nr in [2, 4, 8]:
    for nv in [8, 12, 26]:
        if (nr, nv) != (4, 12):
            gated(per_coin_w, moderate(nr, nv), f'moderate, {nr}w move {nv}w vol')
for L in LOOKBACKS:
    if L != 4:
        gated(signals.per_coin(p, L), on, f'moderate, per coin L{L}')
gated(market_w, on, 'moderate, market L4')
led.save()

table = led.table().set_index('name').loc[gate_names]
frame = led.frame()[table.key]
primary = frame[table.key[table.kind == 'primary']]
neff_primary = gate.effective_trials(primary, extra=OLD_TRIALS)
neff_gate = gate.effective_trials(frame, extra=OLD_TRIALS)
print(f'primary family   {primary.shape[1]} books and {OLD_TRIALS} old trials   effective trials {neff_primary:.1f}')
print(f'gate scoring     {frame.shape[1]} books and {OLD_TRIALS} old trials   effective trials {neff_gate:.1f}')

primary family   32 books and 30 old trials   effective trials 42.1
gate scoring     125 books and 30 old trials   effective trials 73.6


Both counts match, 42.1 for the main test and 73.6 for everything the scoring counted.

## The late check's backtests

Rerun every backtest the late check ran, each at the cost it originally used.

In [3]:
symbols = universe.perps()
bars, prints = data.daily_bars(symbols, '2020-01-01', '2026-09-01'), data.funding(symbols, '2020-01-01', '2026-09-01')
daily_bars = panel.bars(bars, prints)
daily = daily_bars.returns

DAYS = list(range(5, 36))
WEEKS = {'Sun': 'W-SUN', 'Mon': 'W-MON', 'Tue': 'W-TUE', 'Wed': 'W-WED', 'Thu': 'W-THU', 'Fri': 'W-FRI', 'Sat': 'W-SAT'}
trail = {D: daily.rolling(D, min_periods=int(np.ceil(D * 5 / 7))).sum() for D in DAYS}
books = {name: Book(panel.build(daily_bars, rule), costs.Provisional(), ledger=led) for name, rule in WEEKS.items()}
sun = books['Sun']
before = len(led.table())

def at(b, x, week):
    return x.resample(week).last().reindex(b.p.returns.index)

def market_trail(b, D, week):
    liquid = b.p.liquidity.reindex(daily.index, method='bfill') >= 1e7
    return at(b, daily.where(liquid).mean(axis=1).rolling(D, min_periods=D).sum(), week)

for L in (1, 2, 3, 4, 5):
    sun.run(signals.per_coin(sun.p, L), name='late, per coin weekly build', params={'weeks': L})
    sun.run(signals.market(sun.p, L), name='late, market weekly build', params={'weeks': L})
for day, b in books.items():
    for D in DAYS:
        b.run(inverse_vol(b.p, np.sign(at(b, trail[D], WEEKS[day])).shift(1)), name='late, per coin', params={'day': day, 'days': D})
        b.run(signals.basket(b.p).mul(np.sign(market_trail(b, D, WEEKS[day])).shift(1), axis=0), name='late, market', params={'day': day, 'days': D})

seven_days = Sliced(daily_bars, costs.Provisional(), periods=list(WEEKS.values()), ledger=led)
sliced = {D: seven_days.run(lambda q, D=D: inverse_vol(q, np.sign(trail[D].resample(q.period).last().reindex(q.returns.index)).shift(1)),
                            name='late, seven-slice', params={'days': D}).net for D in DAYS}

BLOCK = list(range(25, 34))
level_d = daily.fillna(0.0).cumsum()
clock = pd.Series(np.arange(len(level_d), dtype=float), index=level_d.index)
fit = {D: level_d.rolling(D).corr(clock).where(daily.notna().rolling(D).sum() >= int(np.ceil(D * 5 / 7))) for D in BLOCK}
slope = {D: level_d.rolling(D).cov(clock).div(clock.rolling(D).var(), axis=0).where(fit[D].notna()) for D in BLOCK}
CUTS, Z_CUTS = [0.0, 0.2, 0.4, 0.6, 0.8], [0.0, 0.5, 1.0, 1.5, 2.0]

def thirds_of(x):
    lo, hi = x.quantile(1 / 3, axis=1), x.quantile(2 / 3, axis=1)
    return (x.gt(lo, axis=0).astype(float) + x.ge(hi, axis=0).astype(float)).where(x.notna())

def fits(fit_by_days, tag):
    """The line-fit thirds and fixed cuts on one set of fits."""
    for day, b in books.items():
        for D in BLOCK:
            ret = at(b, trail[D], WEEKS[day])
            corr = at(b, fit_by_days[D], WEEKS[day])
            r2 = (corr ** 2).where((b.p.liquidity.shift(-1) >= 1e7) & b.p.vol.shift(-1).notna() & ret.notna())
            valid = r2.notna()
            choppy = r2.le(r2.quantile(1 / 3, axis=1), axis=0) & valid
            clean = r2.ge(r2.quantile(2 / 3, axis=1), axis=0) & valid
            sign, slope_sign = np.sign(ret), np.sign(corr)
            books_ = [('plain sign', sign), ('sign, choppy third', sign.where(choppy)), ('sign, middle third', sign.where(valid & ~choppy & ~clean)),
                      ('sign, clean third', sign.where(clean))]
            if tag == 'daily':
                books_ += [('slope sign', slope_sign), ('slope sign, clean third', slope_sign.where(clean))]
            for label, sig in books_:
                b.run(inverse_vol(b.p, sig.shift(1)), name=f'late, {tag} fit thirds', params={'day': day, 'days': D, 'book': label})
            raw = (corr ** 2).shift(1)
            directions = [('trailing return', sign), ('slope', slope_sign)] if tag == 'daily' else [('trailing return', sign)]
            for label, sig in directions:
                full = inverse_vol(b.p, sig.shift(1))
                for cut in CUTS:
                    b.run(full.where(raw >= cut, 0.0), name=f'late, {tag} fit cut', params={'day': day, 'days': D, 'direction': label, 'cut': cut})
                    if label == 'trailing return' and cut > 0:
                        b.run(full.where(raw < cut, 0.0), name=f'late, {tag} fit cut, removed coins', params={'day': day, 'days': D, 'cut': cut})

fits(fit, 'daily')

for day, b in books.items():
    for D in BLOCK:
        ret = at(b, trail[D], WEEKS[day])
        ok = (b.p.liquidity.shift(-1) >= 1e7) & b.p.vol.shift(-1).notna() & ret.notna()
        r2 = (at(b, fit[D], WEEKS[day]) ** 2).where(ok)
        line = at(b, slope[D], WEEKS[day])
        z = (line.abs() * D / (b.p.vol.shift(-1) * np.sqrt(D / 7))).where(ok & r2.notna())
        r2 = r2.where(z.notna())
        sign = np.sign(ret)
        steep3, clean3 = thirds_of(z), thirds_of(r2)
        for k in range(3):
            b.run(inverse_vol(b.p, sign.where(steep3 == k).shift(1)), name='late, steepness third', params={'day': day, 'days': D, 'third': k})
            for j in range(3):
                b.run(inverse_vol(b.p, sign.where((steep3 == k) & (clean3 == j)).shift(1)), name='late, steepness and tightness box',
                      params={'day': day, 'days': D, 'steep': k, 'tight': j})
        for label, sig in [('trailing return', sign), ('slope', np.sign(line))]:
            full = inverse_vol(b.p, sig.shift(1))
            for cut in Z_CUTS:
                passes = z >= cut
                kw = {'day': day, 'days': D, 'direction': label, 'cut': cut}
                b.run(full.where(passes.shift(1, fill_value=False), 0.0), name='late, steepness cut', params=kw)
                b.run(inverse_vol(b.p, sig.where(passes).shift(1)), name='late, steepness cut, fully invested', params=kw)
                if cut > 0:
                    b.run(full.where(~passes.shift(1, fill_value=False), 0.0), name='late, steepness cut, removed coins', params=kw)

D = 28
seven_at_cost = Sliced(daily_bars, costs.Taker(costs.FEES['usdm'], ladder, 1e5), periods=list(WEEKS.values()), ledger=led)

def pieces(q):
    ret = trail[D].resample(q.period).last().reindex(q.returns.index)
    ok = (q.liquidity.shift(-1) >= 1e7) & q.vol.shift(-1).notna() & ret.notna()
    line = slope[D].resample(q.period).last().reindex(q.returns.index)
    z = (line.abs() * D / (q.vol.shift(-1) * np.sqrt(D / 7))).where(ok)
    return np.sign(ret), np.sign(line), z

def rule(cut, invested=True):
    def weights(q):
        _, direction, z = pieces(q)
        if invested:
            return inverse_vol(q, direction.where(z >= cut).shift(1))
        return inverse_vol(q, direction.shift(1)).where((z >= cut).shift(1, fill_value=False), 0.0)
    return weights

def rule3(z_cut, r2_cut):
    def weights(q):
        _, direction, z = pieces(q)
        r2 = fit[D].resample(q.period).last().reindex(q.returns.index) ** 2
        return inverse_vol(q, direction.where((z >= z_cut) & (r2 >= r2_cut)).shift(1))
    return weights

plain = seven_at_cost.run(lambda q: inverse_vol(q, pieces(q)[0].shift(1)), name='late, plain sign at cost').net
slopes = [seven_at_cost.run(rule(cut), name='late, slope steepness at cost', params={'cut': cut}).net for cut in Z_CUTS]
for cut in Z_CUTS[1:]:
    seven_at_cost.run(rule(cut, invested=False), name='late, slope steepness at cost, as traded', params={'cut': cut})
Z3, R3 = [0.0, 1.0, 1.5, 2.0], [0.0, 0.4, 0.6, 0.8]
grid3 = {(zc, rc): seven_at_cost.run(rule3(zc, rc), name='late, strict grid', params={'steep': zc, 'r2': rc}).net for zc in Z3 for rc in R3}
start = max(x.index[0] for x in [plain, *slopes])
base, corner = plain.loc[start:], grid3[(2.0, 0.8)].loc[start:]
pair = led.record((base / base.std() + corner / corner.std()) / 2 * base.std(), 'late, the pair')
print(f'daily-fit and steepness books done, {len(led.table()) - before} late check trials so far', flush=True)

daily-fit and steepness books done, 0 late check trials so far


## The hourly fits

The last part of the late check fitted its trend lines on hourly prices. Rerun those backtests too.

In [4]:
parts = []
for i in range(0, len(symbols), 100):
    k = data.klines(symbols[i:i + 100], '2020-01-01', '2026-09-01', '1h')
    parts.append(k.pivot_table(index='open_time', columns='symbol', values='close', aggfunc='last').astype('float32'))
hourly = pd.concat(parts, axis=1).sort_index().rename_axis('hour')
del parts

def hourly_fit(prices, windows):
    """Correlation of hourly log price with time over the trailing days, read at each day's last hour."""
    hours = pd.date_range(prices.index[0].floor('D'), prices.index[-1].ceil('D') - pd.Timedelta(hours=1), freq='h')
    y = np.log(prices.reindex(hours).astype('float64')).values
    seen = ~np.isnan(y)
    y = np.where(seen, y - np.nanmean(y, axis=0), 0.0)
    t = np.arange(len(hours))[:, None] / 24.0
    t = t - t.mean()
    cum = lambda a: np.vstack([np.zeros((1, a.shape[1])), np.cumsum(a, axis=0)])
    sums = [cum(a) for a in (seen * 1.0, y, y * y, seen * t, seen * t * t, y * t)]
    ends = np.arange(24, len(hours) + 1, 24)
    out = {}
    for D in windows:
        first = np.maximum(ends - 24 * D, 0)
        n, sy, syy, st, stt, sty = (a[ends] - a[first] for a in sums)
        with np.errstate(all='ignore'):
            corr = (n * sty - st * sy) / np.sqrt((n * stt - st * st) * (n * syy - sy * sy))
        corr[(n < 24 * D * 5 / 7) | (ends < 24 * D)[:, None]] = np.nan
        out[D] = pd.DataFrame(corr, index=hours[ends - 1].floor('D'), columns=prices.columns)
    return out

parts = [hourly_fit(hourly.iloc[:, i:i + 50].dropna(how='all'), BLOCK) for i in range(0, hourly.shape[1], 50)]
fit_h = {D: pd.concat([q[D] for q in parts], axis=1).reindex(index=daily.index, columns=daily.columns) for D in BLOCK}
del parts, hourly
fits(fit_h, 'hourly')
led.save()
late = led.table()
late = late[late.name.str.startswith('late, ')]
print(f'late check   {len(late)} trials recorded')
print(late.groupby('name').size().to_string())

late check   5101 trials recorded
name
late, daily fit cut                         630
late, daily fit cut, removed coins          252
late, daily fit thirds                      378
late, hourly fit cut                        315
late, hourly fit cut, removed coins         252
late, hourly fit thirds                     252
late, market                                217
late, market weekly build                     5
late, per coin                              217
late, per coin weekly build                   5
late, plain sign at cost                      1
late, seven-slice                            31
late, slope steepness at cost                 5
late, slope steepness at cost, as traded      4
late, steepness and tightness box           567
late, steepness cut                         630
late, steepness cut, fully invested         630
late, steepness cut, removed coins          504
late, steepness third                       189
late, strict grid                            16
l

The late check ran 5,101 backtests, not the 5,700 it said.

## The recount

Count every backtest, then recompute the probability that each of the late check's four quoted Sharpes is real. They were first judged against 74 independent backtests.

In [5]:
late_frame = led.frame()[late.key]
neff_late = gate.effective_trials(late_frame)
neff_lane = led.trials(extra=OLD_TRIALS)
print(f'late check alone   {late_frame.shape[1]} trials   effective trials {neff_late:.1f}')
print(f'whole lane         {late_frame.shape[1] + len(gate_names)} trials and {OLD_TRIALS} old trials   effective trials {neff_lane:.1f}\n')

quoted = {'seven-slice, 28 days, provisional cost': sliced[28], 'plain sign, seven-slice at cost': base,
          'strict corner, steepness 2.0 and R2 0.8': corner, 'the pair': pair}
rows = {}
for k, x in quoted.items():
    row = {'Sharpe': sharpe(x), 'weeks': len(x.dropna())}
    for label, n in [('42', 42), ('74', 74), ('lane', neff_lane)]:
        prob, bar = gate.significance(x, n)
        row[f'probability, {label}'], row[f'luck bar, {label}'] = prob, bar
    rows[k] = row
print(pd.DataFrame(rows).T.round(2).to_string())
print('\nper coin $10M L4, primary', {label: round(gate.significance(led.frame()['per coin $10M L4'].dropna(), n)[0], 2)
                                       for label, n in [('primary', neff_primary), ('lane', neff_lane)]})

late check alone   5101 trials   effective trials 197.5
whole lane         5226 trials and 30 old trials   effective trials 306.5

                                         Sharpe  weeks  probability, 42  luck bar, 42  probability, 74  luck bar, 74  probability, lane  luck bar, lane
seven-slice, 28 days, provisional cost     1.02  337.0             0.67          0.86             0.58          0.94               0.39            1.13
plain sign, seven-slice at cost            1.04  327.0             0.67          0.87             0.58          0.96               0.39            1.15
strict corner, steepness 2.0 and R2 0.8    0.97  327.0             0.63          0.87             0.52          0.96               0.28            1.15
the pair                                   1.22  327.0             0.82          0.87             0.76          0.96               0.57            1.15



per coin $10M L4, primary {'primary': 0.49, 'lane': 0.23}


All 5,226 backtests are worth 306.5 independent ones, four times the 74 used before.

| result | Sharpe | probability real, against 74 | against 306.5 |
|---|---|---|---|
| seven-slice, 28 days | 1.02 | 0.58 | 0.39 |
| plain sign at the measured cost | 1.04 | 0.58 | 0.39 |
| strict corner | 0.97 | 0.52 | 0.28 |
| the pair | 1.22 | 0.76 | 0.57 |

Only the pair stays above the 0.5 park line. The main 4-week result stays at 0.49, since it is judged only against the main test's own backtests.